# Week 1-Data Preprocessing

In [1]:
%pip install pandas numpy matplotlib seaborn openpyxl

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print("Libraries loaded successfully!")

Libraries loaded successfully!


In [3]:
import os

print(os.getcwd())
print(os.listdir())

C:\Users\DELL\Devnexes_Data_Analysis_Internship
['.ipynb_checkpoints', 'app.py', 'dir', 'E-Commerce cutomer dashboard.jpeg', 'Online Retail.xlsx', 'rfm_data.csv', 'streamlit', 'Untitled.ipynb']


In [4]:
df = pd.read_excel("Online Retail.xlsx")

print("Dataset loaded successfully!")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

Dataset loaded successfully!
Rows: 541909
Columns: 8


In [5]:
print(df.columns.tolist())

['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country']


In [6]:
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  object        
 1   StockCode    541909 non-null  object        
 2   Description  540455 non-null  object        
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[us]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      541909 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), object(3), str(1)
memory usage: 40.0+ MB


In [8]:
print("Duplicate rows:", df.duplicated().sum())

Duplicate rows: 5268


In [9]:
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])

print("InvoiceDate type:", df["InvoiceDate"].dtype)

InvoiceDate type: datetime64[us]


In [10]:
cancelled = df["InvoiceNo"].astype(str).str.startswith("C")

print("Cancelled orders:", cancelled.sum())

Cancelled orders: 9288


In [11]:
df = df[~cancelled].copy()

print("Remaining records:", len(df))

Remaining records: 532621


In [12]:
df["Revenue"] = df["Quantity"] * df["UnitPrice"]

df[["Quantity", "UnitPrice", "Revenue"]].head()

,Quantity,UnitPrice,Revenue
0,6,2.55,15.30
1,6,3.39,20.34
2,8,2.75,22.00
3,6,3.39,20.34
4,6,3.39,20.34


In [13]:
df["Month"] = df["InvoiceDate"].dt.to_period("M")

df[["InvoiceDate", "Month"]].head()

,InvoiceDate,Month
0,2010-12-01 08:26:00,2010-12
1,2010-12-01 08:26:00,2010-12
2,2010-12-01 08:26:00,2010-12
3,2010-12-01 08:26:00,2010-12
4,2010-12-01 08:26:00,2010-12


In [14]:
monthly_revenue = (
    df.groupby("Month")["Revenue"]
      .sum()
      .reset_index()
)

monthly_revenue

,Month,Revenue
0,2010-12,823746.140
1,2011-01,691364.560
2,2011-02,523631.890
3,2011-03,717639.360
4,2011-04,537808.621
5,2011-05,770536.020
6,2011-06,761739.900
7,2011-07,719221.191
8,2011-08,737014.260
9,2011-09,1058590.172


In [15]:
df.isnull().sum()

InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     134697
Country             0
Revenue             0
Month               0
dtype: int64

In [16]:
print("Quantity:")
print(df["Quantity"].describe())

print("\nUnitPrice:")
print(df["UnitPrice"].describe())

print("\nRevenue:")
print(df["Revenue"].describe())

Quantity:
count    532621.000000
mean         10.239972
std         159.593551
min       -9600.000000
25%           1.000000
50%           3.000000
75%          10.000000
max       80995.000000
Name: Quantity, dtype: float64

UnitPrice:
count    532621.000000
mean          3.847621
std          41.758023
min      -11062.060000
25%           1.250000
50%           2.080000
75%           4.130000
max       13541.330000
Name: UnitPrice, dtype: float64

Revenue:
count    532621.000000
mean         19.985244
std         270.574241
min      -11062.060000
25%           3.750000
50%           9.900000
75%          17.700000
max      168469.600000
Name: Revenue, dtype: float64


In [17]:
# Remove invalid negative values
df = df[(df["Quantity"] > 0) & (df["UnitPrice"] > 0)].copy()

# IQR method for extreme outliers
Q1 = df[["Quantity", "UnitPrice", "Revenue"]].quantile(0.25)
Q3 = df[["Quantity", "UnitPrice", "Revenue"]].quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

df = df[
    (df["Quantity"] >= lower_bound["Quantity"]) &
    (df["Quantity"] <= upper_bound["Quantity"]) &
    (df["UnitPrice"] >= lower_bound["UnitPrice"]) &
    (df["UnitPrice"] <= upper_bound["UnitPrice"]) &
    (df["Revenue"] >= lower_bound["Revenue"]) &
    (df["Revenue"] <= upper_bound["Revenue"])
].copy()

print("Outliers handled successfully!")
print("Remaining records:", len(df))

Outliers handled successfully!
Remaining records: 424366


In [18]:
# Handle missing CustomerID
df = df.dropna(subset=["CustomerID"]).copy()

# Handle missing Description
df["Description"] = df["Description"].fillna("Unknown")

# Check missing values again
print(df.isnull().sum())

InvoiceNo      0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
UnitPrice      0
CustomerID     0
Country        0
Revenue        0
Month          0
dtype: int64


In [19]:
monthly_revenue = (
    df.groupby("Month")["Revenue"]
      .sum()
      .reset_index()
)

monthly_revenue

,Month,Revenue
0,2010-12,210596.030
1,2011-01,177632.100
2,2011-02,172035.850
3,2011-03,228254.080
4,2011-04,191731.201
5,2011-05,257416.280
6,2011-06,227627.380
7,2011-07,217639.371
8,2011-08,237534.580
9,2011-09,359736.742


# WEEK 2 — RFM Customer Segmentation

In [20]:
analysis_date = df["InvoiceDate"].max() + pd.Timedelta(days=1)

print("Analysis Date:", analysis_date)

Analysis Date: 2011-12-10 12:50:00


In [21]:
rfm = df.groupby("CustomerID").agg(
    Recency=("InvoiceDate", lambda x: (analysis_date - x.max()).days),
    Frequency=("InvoiceNo", "nunique"),
    Monetary=("Revenue", "sum")
).reset_index()

rfm.head()

,CustomerID,Recency,Frequency,Monetary
0,12347.0,2,7,2430.57
1,12348.0,249,1,17.00
2,12349.0,19,1,997.35
3,12350.0,310,1,274.00
4,12352.0,36,7,1147.44


In [22]:
rfm["R_Score"] = pd.qcut(
    rfm["Recency"],
    5,
    labels=[5, 4, 3, 2, 1]
)

rfm["F_Score"] = pd.qcut(
    rfm["Frequency"].rank(method="first"),
    5,
    labels=[1, 2, 3, 4, 5]
)

rfm["M_Score"] = pd.qcut(
    rfm["Monetary"].rank(method="first"),
    5,
    labels=[1, 2, 3, 4, 5]
)

rfm["RFM_Score"] = (
    rfm["R_Score"].astype(str)
    + rfm["F_Score"].astype(str)
    + rfm["M_Score"].astype(str)
)

rfm.head()

,CustomerID,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,RFM_Score
0,12347.0,2,7,2430.57,5,5,5,555
1,12348.0,249,1,17.00,1,1,1,111
2,12349.0,19,1,997.35,4,1,4,414
3,12350.0,310,1,274.00,1,1,2,112
4,12352.0,36,7,1147.44,3,5,5,355


In [23]:
def segment_customer(row):
    if row["R_Score"] >= 4 and row["F_Score"] >= 4 and row["M_Score"] >= 4:
        return "Champions"
    elif row["R_Score"] <= 2 and row["F_Score"] <= 2:
        return "Churned"
    else:
        return "Regular"

rfm["Segment"] = rfm.apply(segment_customer, axis=1)

rfm.head()

,CustomerID,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,RFM_Score,Segment
0,12347.0,2,7,2430.57,5,5,5,555,Champions
1,12348.0,249,1,17.00,1,1,1,111,Churned
2,12349.0,19,1,997.35,4,1,4,414,Regular
3,12350.0,310,1,274.00,1,1,2,112,Churned
4,12352.0,36,7,1147.44,3,5,5,355,Regular


In [24]:
segment_counts = rfm["Segment"].value_counts()

print(segment_counts)

Segment
Regular      2201
Churned      1016
Champions     905
Name: count, dtype: int64


In [25]:
rfm.to_csv("rfm_data.csv", index=False)

print("RFM data saved successfully!")

RFM data saved successfully!


In [26]:
import os

print(os.path.exists("rfm_data.csv"))

True


# WEEK 4: BUSINESS INSIGHTS REPORT

## Executive Summary

This project analyzed an e-commerce sales dataset to understand customer purchasing behavior and retention.

The analysis included data preprocessing, RFM customer segmentation, and monthly cohort retention analysis. A total of 4,122 customers were segmented into three groups: 905 Champions, 2,201 Regular customers, and 1,016 Churned customers.

The results show that Regular customers represent the largest customer group, while a significant number of customers are classified as Churned. This indicates an opportunity to improve customer retention and encourage repeat purchases.

The monthly cohort retention analysis also shows that customer retention varies across cohorts and generally decreases as customers move further from their first purchase. The cohort retention matrix provides a useful view of how customer engagement changes over time.

## Key Customer Retention Findings

1. The analysis identified 4,122 customers in total.

2. Champions: 905 customers were classified as Champions. These customers show strong recent, frequent, and valuable purchasing behavior.

3. Regular Customers: 2,201 customers were classified as Regular. This is the largest segment and represents an important opportunity for increasing customer loyalty and purchase frequency.

4. Churned Customers: 1,016 customers were classified as Churned. This indicates a significant group of customers that may require re-engagement strategies.

5. Cohort retention varies by month. The retention heatmap shows differences in customer retention across monthly cohorts and provides insight into long-term customer engagement.

6. The December 2010 cohort shows 36.6% retention in month 2 and 32.3% in month 3, demonstrating that a portion of customers continued purchasing after their initial purchase.

## Business Recommendations

1. Focus on retaining Champions by providing loyalty rewards, exclusive offers, and personalized promotions.

2. Target Regular customers with personalized offers and product recommendations to encourage more frequent purchases and move them toward the Champions segment.

3. Launch re-engagement campaigns for Churned customers using discounts, reminders, and targeted promotional messages.

4. Monitor monthly cohort retention regularly to identify cohorts with declining retention and investigate possible causes.

5. Use customer segmentation to create targeted marketing strategies instead of using the same promotion for all customers.